# Chapter 3 - Multi-Class Classification

Binary classification says *yes/no*. Multi-class says *which one of k classes?*.

We'll predict the topic of Reuters newswires: 8,982 training / 2,246 test articles, each assigned to one of 46 mutually exclusive topics (classes).

## The three-phase recipe
1. **Vectorize** text into a 10,000-dim bag of words.
2. **One-hot** the integer labels into a 46-dim binary vector (or use sparse categorical cross-entropy with integer labels).
3. Build a network ending in a 46-unit `softmax` layer.

In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import numpy as np
import matplotlib.pyplot as plt

import keras
from keras.datasets import reuters
from keras import layers, models

print("Keras", keras.__version__)

Keras 3.15.1


## Load & prepare (with sensible caps)
`num_words=10000` truncates vocabulary. We also one-hot the labels the manual way first to *see* what it does (use `to_categorical` for storage later).

In [2]:
from keras.datasets import reuters

(train_data, train_labels), (test_data, test_labels) = reuters.load_data(num_words=10000)

print("train", len(train_data), "test", len(test_data), "classes", len(set(train_labels)))

train 8982 test 2246 classes 46


In [3]:
def vectorize_sequences(sequences, dimension=10000):
    results = np.zeros((len(sequences), dimension))
    for i, seq in enumerate(sequences):
        # histogram-like: count word presence
        results[i, seq] = 1.
    return results

x_train = vectorize_sequences(train_data)
x_test = vectorize_sequences(test_data)
print("x_train", x_train.shape)

x_train (8982, 10000)


In [4]:
from keras.utils import to_categorical

y_train = to_categorical(train_labels)
y_test = to_categorical(test_labels)
print("one-hot label vector for first sample:", y_train[0], "shape", y_train.shape)

one-hot label vector for first sample: [0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.] shape (8982, 46)


## The model: 64 -> 64 -> softmax(46)
A wide hidden layer (64 units always more than 46 classes) so the network has room to spread information before the final softmax.

In [5]:
model = models.Sequential([
    layers.Input(shape=(10000,)),
    layers.Dense(64, activation="relu"),
    layers.Dense(64, activation="relu"),
    layers.Dense(46, activation="softmax"),
])
model.compile(optimizer="rmsprop", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │       640,064 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 46)             │         2,990 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 647,214 (2.47 MB)

 Trainable params: 647,214 (2.47 MB)

 Non-trainable params: 0 (0.00 B)

In [6]:
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=15, batch_size=512, verbose=1,
)

Epoch 1/15


 1/18 ━━━━━━━━━━━━━━━━━━━━ 12s 711ms/step - accuracy: 0.0273 - loss: 3.8313

 8/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3662 - loss: 3.2499   

16/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4711 - loss: 2.7230

18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.4849 - loss: 2.6406 - val_accuracy: 0.6331 - val_loss: 1.8101


...

Epoch 15/15


 1/18 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - accuracy: 0.9531 - loss: 0.2020

 8/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9543 - loss: 0.1724 

16/18 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.9508 - loss: 0.1823

18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.9505 - loss: 0.1817 - val_accuracy: 0.7965 - val_loss: 0.9812


## Evaluate on the test set and inspect per-class quality

In [7]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f}  (46 classes -> random is {1/46:.3f})")

Test accuracy: 0.7965  (46 classes -> random is 0.022)


## Classification: predictions are *distributions*, not answers
`predict` on one sample gives 46 probabilities; argmax tells the class, but the shape of the distribution tells how confident the model is.

In [8]:
pred = model.predict(x_train[0:1])
print("pred.shape", pred.shape)
top3 = np.argsort(pred[0])[::-1][:3]
print("top-3 predicted class ids:", top3)
print("true class:", train_labels[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


pred.shape (1, 46)
top-3 predicted class ids: [ 3  4 19]
true class: 3


## Recap
- Multi-class = `softmax` over k outputs, `categorical_crossentropy`.
- One-hot labels are standard; keras's `to_categorical` does it for integer labels.
- Must choose layer sizes >= number of classes to carry enough information.